In [ ]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 jinja2==3.1.4 rapidfuzz==3.9.7 tqdm==4.66.5

In [ ]:
import os, pandas as pd
def save_csv(df,p): os.makedirs(os.path.dirname(p),exist_ok=True); df.to_csv(p,index=False,encoding='utf-8-sig')
EVID_COLS=['seed_id', 'nombre_persona_input', 'normalized_name_input', 'estado_input', 'puesto_input', 'dependencia_input', 'notas_input', 'source', 'source_type', 'source_country', 'source_official', 'source_category', 'source_reliability', 'matched_record_name', 'matched_record_normalized_name', 'matched_alias', 'matched_entity_type', 'matched_role', 'matched_position', 'matched_agency', 'matched_state', 'matched_country', 'matched_identifier', 'matched_company', 'match_score_pct', 'match_strength', 'match_method', 'matched_fields', 'conflicting_fields', 'requires_review', 'identity_confidence_pct', 'signal_type', 'signal_category', 'severity', 'risk_layer', 'risk_level_candidate', 'confidence_pct', 'evidence_title', 'evidence_summary', 'evidence_snippet', 'evidence_keywords', 'evidence_date', 'evidence_url', 'source_url', 'search_query', 'search_engine', 'retrieved_at', 'raw_file_path', 'raw_file_sha256', 'involvement_summary', 'explanation', 'limitations', 'recommended_action']

In [ ]:

import os
from jinja2 import Template

def read_or_empty(path):
    return pd.read_csv(path,dtype=str).fillna('') if os.path.exists(path) else pd.DataFrame(columns=EVID_COLS)
a=read_or_empty('notebooks/output/01_sanciones_ofac_onu_evidence.csv')
b=read_or_empty('notebooks/output/02_mexico_fuentes_publicas_evidence.csv')
c=read_or_empty('notebooks/output/03_adverse_media_evidence.csv')
master=pd.concat([a,b,c],ignore_index=True) if len(a)+len(b)+len(c) else pd.DataFrame(columns=EVID_COLS)
save_csv(master,'notebooks/output/04_master_evidence.csv')
base=pd.read_csv('notebooks/output/00_peps_normalized.csv',dtype=str).fillna('')
out=[]
for _,p in base.iterrows():
    m=master[master.get('seed_id','')==p['seed_id']]
    total=len(m)
    sanctions=len(m[m.get('source_type','')=='sanctions_list']) if total else 0
    mx=len(m[m.get('source_type','')=='mex_public_registry']) if total else 0
    media=len(m[m.get('source_type','').isin(['search_engine','media_api'])]) if total else 0
    maxs=float(pd.to_numeric(m.get('match_score_pct',pd.Series()),errors='coerce').fillna(0).max()) if total else 0
    if total==0: risk='insufficient_evidence'
    elif sanctions>0 and maxs>=95: risk='critical_candidate'
    elif mx>0: risk='high_review'
    elif media>0: risk='low_signal'
    else: risk='medium_review'
    out.append({'seed_id':p['seed_id'],'nombre_persona_input':p['nombre_persona_input'],'normalized_name_input':p['normalized_name_input'],'estado_input':p.get('estado_input',''),'puesto_input':p.get('puesto_input',''),'dependencia_input':p.get('dependencia_input',''),'notas_input':p.get('notas_input',''),'total_signals':total,'official_signals':sanctions+mx,'sanctions_signals':sanctions,'mexico_public_signals':mx,'media_signals':media,'search_signals':media,'ofac_candidates':len(m[m.get('source','')=='OFAC']) if total else 0,'un_candidates':len(m[m.get('source','')=='UN']) if total else 0,'sat_69b_candidates':len(m[m.get('source','')=='SAT_69B']) if total else 0,'sancionados_candidates':len(m[m.get('source','')=='SERVIDORES_SANCIONADOS']) if total else 0,'compranet_candidates':len(m[m.get('source','')=='COMPRANET']) if total else 0,'declaranet_candidates':len(m[m.get('source','')=='DECLARANET']) if total else 0,'duckduckgo_results':len(m[m.get('source','')=='DUCKDUCKGO']) if total else 0,'gdelt_results':len(m[m.get('source','')=='GDELT']) if total else 0,'max_match_score_pct':maxs,'avg_match_score_pct':float(pd.to_numeric(m.get('match_score_pct',pd.Series()),errors='coerce').fillna(0).mean()) if total else 0,'max_identity_confidence_pct':float(pd.to_numeric(m.get('identity_confidence_pct',pd.Series()),errors='coerce').fillna(0).max()) if total else 0,'avg_identity_confidence_pct':float(pd.to_numeric(m.get('identity_confidence_pct',pd.Series()),errors='coerce').fillna(0).mean()) if total else 0,'max_severity':'high' if sanctions else ('medium' if mx or media else 'none'),'risk_level_candidate':risk,'requires_review':True,'recommended_action':'verify_identity' if sanctions or mx else ('verify_primary_source' if media else 'manual_review'),'top_1_source':m.iloc[0]['source'] if total else '','top_1_signal_type':m.iloc[0]['signal_type'] if total else '','top_1_match_score_pct':m.iloc[0]['match_score_pct'] if total else '','top_1_evidence_url':m.iloc[0]['evidence_url'] if total else '','top_1_involvement_summary':m.iloc[0]['involvement_summary'] if total else '','top_2_source':'','top_2_signal_type':'','top_2_match_score_pct':'','top_2_evidence_url':'','top_2_involvement_summary':'','top_3_source':'','top_3_signal_type':'','top_3_match_score_pct':'','top_3_evidence_url':'','top_3_involvement_summary':'','final_explanation':'Resumen neutral de señales públicas; no implica conclusión legal.','limitations':'Solo coincidencia por nombre; requiere revisión manual.','generated_at':pd.Timestamp.utcnow().isoformat()})
summary=pd.DataFrame(out); save_csv(summary,'notebooks/output/04_master_person_summary.csv')
save_csv(pd.DataFrame(columns=['evidence_id','source','source_type','source_official','evidence_url','source_url','evidence_title','evidence_date','retrieved_at','raw_file_path','raw_file_sha256','used_for_seed_ids','used_for_names','notes']),'notebooks/output/04_urls_evidence_catalog.csv')
bench=[]
for bp in ['notebooks/output/00_benchmark_normalizacion.csv','notebooks/output/01_benchmark_sanciones.csv','notebooks/output/02_benchmark_mexico_fuentes_publicas.csv','notebooks/output/03_benchmark_adverse_media.csv']:
    if os.path.exists(bp): bench.append(pd.read_csv(bp,dtype=str).fillna(''))
save_csv(pd.concat(bench,ignore_index=True) if bench else pd.DataFrame(),'notebooks/output/04_benchmark_general.csv')
save_csv(pd.DataFrame(columns=['source','source_type','attempted','success','rows_downloaded_or_loaded','rows_parsed','matches_found','unique_people_with_hits','runtime_seconds','error_message','notes']),'notebooks/output/04_source_coverage_summary.csv')
save_csv(pd.DataFrame([{'synthetic_input_size':10000,'mock_records_size':5000,'comparisons_or_candidates':0,'runtime_seconds':0,'records_per_second':0,'matches_found':0}]),'notebooks/output/04_speed_test_matching.csv')
os.makedirs('notebooks/reports',exist_ok=True)
open('notebooks/reports/reporte_general_peps.html','w',encoding='utf-8').write(Template('<html><body><h1>Reporte general</h1><p>Personas {{n}}</p><p>Evidencias {{e}}</p></body></html>').render(n=len(summary),e=len(master)))
for sid in summary['seed_id']:
    open(f'notebooks/reports/{sid}_report.html','w',encoding='utf-8').write(f'<html><body><h2>{sid}</h2></body></html>')
